
# Thesis-Ready Multimodal Sentiment Analysis
## Uncertainty-Aware Co-Attention + Evidential Deep Learning (EDL)

Dataset:
- `D:/MVSA_SINGLE`


In [ ]:

# ============================================================
# CONFIGURATION
# ============================================================

class CFG:

    # =========================
    # PATH
    # =========================
    ROOT_DIR = r"D:/MVSA_SINGLE"
    DATA_DIR = r"D:/MVSA_SINGLE/data"
    LABEL_PATH = r"D:/MVSA_SINGLE/labelResultAllFinal.txt"


    # =========================
    # DEVICE
    # =========================
    DEVICE = "cuda"


In [14]:
import pandas as pd
import os
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(CFG.LABEL_PATH, header=0, sep=',')
df.columns = ["id", "text_label", "image_label", "final_label"]

def is_valid(row):

    if row["text_label"] == "positive" and row["image_label"] == "negative":
        return False

    if row["text_label"] == "negative" and row["image_label"] == "positive":
        return False

    return True

df = df[df.apply(is_valid, axis=1)]
df = df.reset_index(drop=True)

print(f"Dataset size after filtering: {len(df)}")

label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

df["label"] = df["final_label"].map(label_map)

# Load text file dengan better error handling
def load_text(sample_id):
    path = os.path.join(CFG.DATA_DIR, f"{sample_id}.txt")

    encodings = ["utf-8", "latin-1", "cp1252", "iso-8859-1"]

    for encoding in encodings:
        try:
            with open(path, "r", encoding=encoding) as f:
                text = f.read().strip()
                if text:  # Jika text berhasil dibaca dan tidak kosong
                    return text
        except FileNotFoundError:
            continue
        except Exception as e:
            continue

    # Jika semua encoding gagal atau file tidak ada
    return ""

# Track failed samples for debugging
failed_samples = []

df["text"] = df["id"].apply(load_text)

# Hitung empty text
empty_text_count = (df["text"] == "").sum()
print(f"\n{'='*60}")
print(f"PREPROCESSING STATISTICS:")
print(f"{'='*60}")
print(f"Total samples: {len(df)}")
print(f"Samples with EMPTY text: {empty_text_count}")
print(f"Samples with VALID text: {len(df) - empty_text_count}")
print(f"Percentage of empty text: {(empty_text_count/len(df)*100):.2f}%")
print(f"{'='*60}\n")

if empty_text_count > 0:
    print("IDs with empty text:")
    empty_ids = df[df["text"] == ""]["id"].tolist()
    for idx in empty_ids[:10]:  # Show first 10
        print(f"  - {idx}")
    if len(empty_ids) > 10:
        print(f"  ... and {len(empty_ids) - 10} more")
    print()

# image path
df["image_path"] = df["id"].apply(
    lambda x: os.path.join(CFG.DATA_DIR, f"{x}.jpg")
)

df.head()


Dataset size after filtering: 4511

PREPROCESSING STATISTICS:
Total samples: 4511
Samples with EMPTY text: 0
Samples with VALID text: 4511
Percentage of empty text: 0.00%



,id,text_label,image_label,final_label,label,text,image_path
0,1,neutral,positive,positive,2,How I feel today #legday #jelly #aching #gym,D:/MVSA_SINGLE/data\1.jpg
1,2,neutral,positive,positive,2,grattis min griskulting!!!???? va bara tvungen...,D:/MVSA_SINGLE/data\2.jpg
2,3,neutral,positive,positive,2,RT @polynminion: The moment I found my favouri...,D:/MVSA_SINGLE/data\3.jpg
3,4,positive,positive,positive,2,#escort We have a young and energetic team and...,D:/MVSA_SINGLE/data\4.jpg
4,5,positive,positive,positive,2,"RT @chrisashaffer: Went to SSC today to be a ""...",D:/MVSA_SINGLE/data\5.jpg


## ⚙️ Fase 1: Imports, Extended Configuration & Data Pipeline

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoImageProcessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from PIL import Image
import numpy as np
import random
import copy
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# EXTENDED CONFIGURATION
# ============================================================
# Tambahan config ke CFG yang sudah ada
CFG.TEXT_MODEL = "roberta-base"
CFG.IMAGE_MODEL = "google/vit-base-patch16-224"
CFG.MAX_LEN = 128
CFG.IMG_SIZE = 224
CFG.BATCH_SIZE = 16
CFG.EPOCHS = 15
CFG.LR_BACKBONE = 2e-5
CFG.LR_HEAD = 1e-3
CFG.WEIGHT_DECAY = 1e-2
CFG.NUM_CLASSES = 3
CFG.FEATURE_DIM = 768
CFG.HIDDEN_DIM = 256
CFG.NUM_HEADS = 4
CFG.DROPOUT = 0.3
CFG.ANNEALING_STEP = 10
CFG.SEED = 42
CFG.NUM_WORKERS = 2
CFG.FREEZE_EPOCHS = 3
CFG.LAMBDA_ORTH = 0.1

# Seed for reproducibility
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG.SEED)
device = torch.device(CFG.DEVICE if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# TOKENIZER & IMAGE PROCESSOR
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(CFG.TEXT_MODEL)
image_processor = AutoImageProcessor.from_pretrained(CFG.IMAGE_MODEL)
print(f"Text model: {CFG.TEXT_MODEL}")
print(f"Image model: {CFG.IMAGE_MODEL}")

In [ ]:
# ============================================================
# DATASET CLASS (Step 1.2)
# ============================================================
class MVSADataset(Dataset):
    def __init__(self, dataframe, tokenizer, image_processor, max_len=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"]) if row["text"] else ""

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        try:
            image = Image.open(row["image_path"]).convert("RGB")
        except Exception:
            image = Image.new("RGB", (CFG.IMG_SIZE, CFG.IMG_SIZE), (0, 0, 0))

        pixel_values = self.image_processor(
            image, return_tensors="pt"
        )["pixel_values"]

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "pixel_values": pixel_values.squeeze(0),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

# ============================================================
# TRAIN / VAL / TEST SPLIT (Step 1.3)
# ============================================================
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=CFG.SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=CFG.SEED, stratify=temp_df["label"]
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train label dist:\n{train_df['label'].value_counts().sort_index()}")

train_dataset = MVSADataset(train_df, tokenizer, image_processor, CFG.MAX_LEN)
val_dataset   = MVSADataset(val_df, tokenizer, image_processor, CFG.MAX_LEN)
test_dataset  = MVSADataset(test_df, tokenizer, image_processor, CFG.MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

# Quick sanity check
batch = next(iter(train_loader))
print(f"\nSanity check batch shapes:")
print(f"  input_ids:    {batch['input_ids'].shape}")
print(f"  attention_mask: {batch['attention_mask'].shape}")
print(f"  pixel_values: {batch['pixel_values'].shape}")
print(f"  labels:       {batch['label'].shape}")

## 🧠 Fase 2: Unimodal Feature Extraction Module
Mengekstraksi fitur sequence dari kedua modalitas menggunakan model pre-trained.
- **Text Encoder:** RoBERTa → sequence hidden states → Linear projection
- **Image Encoder:** ViT → patch hidden states → Linear projection

In [ ]:
# ============================================================
# TEXT ENCODER (Step 2.1)
# ============================================================
class TextEncoder(nn.Module):
    def __init__(self, model_name, feature_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state          # (B, L, 768)
        projected = self.projection(hidden_states)          # (B, L, hidden_dim)
        return projected, attention_mask

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


# ============================================================
# IMAGE ENCODER (Step 2.2)
# ============================================================
class ImageEncoder(nn.Module):
    def __init__(self, model_name, feature_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.projection = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values)
        hidden_states = outputs.last_hidden_state          # (B, N+1, 768)
        projected = self.projection(hidden_states)          # (B, N+1, hidden_dim)
        return projected  # no padding mask needed for ViT

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


print("✅ TextEncoder & ImageEncoder defined.")

## ✂️ Fase 3: Feature Decoupling Module
Memisahkan fitur menjadi representasi **sentimen** dan **latar belakang (background)**.
- Dua MLP per modalitas (sentiment & background)
- Orthogonal Loss memaksa kedua representasi saling ortogonal

In [ ]:
# ============================================================
# FEATURE DECOUPLER (Step 3.1 - 3.2)
# ============================================================
class FeatureDecoupler(nn.Module):
    """Memisahkan fitur menjadi komponen sentimen dan background."""
    def __init__(self, hidden_dim, dropout=0.1):
        super().__init__()
        self.sentiment_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.background_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, features):
        """
        Args:
            features: (B, L, d) sequence features
        Returns:
            sent: (B, L, d) sentiment features
            bg:   (B, L, d) background features
        """
        sent = self.sentiment_mlp(features)
        bg = self.background_mlp(features)
        return sent, bg

    @staticmethod
    def orthogonal_loss(sent, bg):
        """Menghitung orthogonal loss agar sent ⊥ bg."""
        # Mean pool over sequence dimension first
        sent_pooled = sent.mean(dim=1)  # (B, d)
        bg_pooled = bg.mean(dim=1)      # (B, d)
        sent_norm = F.normalize(sent_pooled, dim=-1)
        bg_norm = F.normalize(bg_pooled, dim=-1)
        return torch.mean(torch.abs(torch.sum(sent_norm * bg_norm, dim=-1)))


print("✅ FeatureDecoupler defined.")

## 🔄 Fase 4: Bi-directional Cross-Attention (BCA) Module
Fusi asimetris dua arah antara modalitas teks dan gambar.
- **T→V:** Teks sebagai Query, Gambar sebagai Key/Value
- **V→T:** Gambar sebagai Query, Teks sebagai Key/Value

In [ ]:
# ============================================================
# BI-DIRECTIONAL CROSS-ATTENTION (Step 4.1 - 4.2)
# ============================================================
class CrossAttentionBlock(nn.Module):
    """Single-direction cross-attention with residual connection."""
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        self.norm = nn.LayerNorm(hidden_dim)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, query, key_value, key_padding_mask=None):
        """
        Args:
            query:     (B, Lq, d)
            key_value: (B, Lk, d)
            key_padding_mask: (B, Lk) True = ignore
        Returns:
            output: (B, Lq, d)
        """
        attn_out, _ = self.attention(
            query, key_value, key_value,
            key_padding_mask=key_padding_mask
        )
        x = self.norm(query + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x


class BiDirectionalCrossAttention(nn.Module):
    """Bi-directional cross-attention antara teks dan gambar."""
    def __init__(self, hidden_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.t2v = CrossAttentionBlock(hidden_dim, num_heads, dropout)
        self.v2t = CrossAttentionBlock(hidden_dim, num_heads, dropout)

    def forward(self, text_feat, image_feat, text_mask=None):
        """
        Args:
            text_feat:  (B, Lt, d) - sentiment features dari teks
            image_feat: (B, Li, d) - sentiment features dari gambar
            text_mask:  (B, Lt) attention mask (1=valid, 0=pad)
        Returns:
            attn_t2v: (B, d) - teks menyoroti gambar (pooled)
            attn_v2t: (B, d) - gambar menyoroti teks (pooled)
        """
        # Invert mask for nn.MultiheadAttention: True = ignore
        text_key_pad_mask = None
        if text_mask is not None:
            text_key_pad_mask = (text_mask == 0)

        # T→V: text queries, image keys/values
        out_t2v = self.t2v(text_feat, image_feat)            # (B, Lt, d)
        # V→T: image queries, text keys/values
        out_v2t = self.v2t(image_feat, text_feat, text_key_pad_mask)  # (B, Li, d)

        # Mean pooling
        if text_mask is not None:
            mask_expanded = text_mask.unsqueeze(-1).float()   # (B, Lt, 1)
            pooled_t2v = (out_t2v * mask_expanded).sum(1) / mask_expanded.sum(1).clamp(min=1)
        else:
            pooled_t2v = out_t2v.mean(dim=1)

        pooled_v2t = out_v2t.mean(dim=1)                     # (B, d)

        return pooled_t2v, pooled_v2t


print("✅ BiDirectionalCrossAttention defined.")

## ⚖️ Fase 5: Evidential Deep Learning (EDL) & Adaptive Gating
Menghitung ketidakpastian epistemik dari setiap aliran cross-attention.
- **Evidence** = Softplus(Linear(features))
- **Dirichlet:** α = e + 1, S = Σα
- **Uncertainty:** u = K/S
- **Gating:** F' = A · (1 - u)

In [ ]:
# ============================================================
# EVIDENTIAL DEEP LEARNING LAYER (Step 5.1 - 5.4)
# ============================================================
class EvidentialLayer(nn.Module):
    """Menghasilkan evidence, belief, dan uncertainty dari fitur."""
    def __init__(self, hidden_dim, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.evidence_layer = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        self.softplus = nn.Softplus()

    def forward(self, features):
        """
        Args:
            features: (B, d) pooled features
        Returns:
            dict with evidence, alpha, belief, uncertainty
        """
        # Step 5.1: Evidence (non-negative)
        evidence = self.softplus(self.evidence_layer(features))  # (B, K)

        # Step 5.2: Dirichlet parameters
        alpha = evidence + 1.0                                    # (B, K)
        S = alpha.sum(dim=-1, keepdim=True)                      # (B, 1)

        # Step 5.3: Belief & Uncertainty
        belief = (alpha - 1.0) / S                               # (B, K)
        uncertainty = self.num_classes / S                        # (B, 1)

        return {
            "evidence": evidence,
            "alpha": alpha,
            "S": S,
            "belief": belief,
            "uncertainty": uncertainty
        }


print("✅ EvidentialLayer defined.")

## 🔗 Fase 6: Dempster-Shafer Evidential Fusion
Menggabungkan belief mass dari dua aliran cross-attention menggunakan aturan Dempster-Shafer.

In [ ]:
# ============================================================
# DEMPSTER-SHAFER FUSION (Step 6.1 - 6.2)
# ============================================================
def dempster_shafer_fusion(belief1, u1, belief2, u2):
    """
    Menggabungkan dua opini subjektif menggunakan Dempster's Rule.

    Args:
        belief1: (B, K) belief mass dari aliran 1
        u1:      (B, 1) uncertainty dari aliran 1
        belief2: (B, K) belief mass dari aliran 2
        u2:      (B, 1) uncertainty dari aliran 2
    Returns:
        fused_belief: (B, K)
        fused_uncertainty: (B, 1)
    """
    K = belief1.shape[-1]

    # Hitung conflict mass C
    # C = sum of b1_i * b2_j for i != j
    b1_expand = belief1.unsqueeze(2)   # (B, K, 1)
    b2_expand = belief2.unsqueeze(1)   # (B, 1, K)
    cross = b1_expand * b2_expand      # (B, K, K)

    # Conflict = off-diagonal sum
    mask = 1.0 - torch.eye(K, device=belief1.device).unsqueeze(0)
    C = (cross * mask).sum(dim=[1, 2])  # (B,)
    C = C.unsqueeze(1)                  # (B, 1)

    # Normalization factor
    norm = (1.0 - C).clamp(min=1e-8)

    # Fused belief: (b1*b2 + b1*u2 + u1*b2) / (1-C)
    b1_b2 = (belief1 * belief2)         # (B, K) element-wise agreement
    b1_u2 = belief1 * u2                # (B, K)
    u1_b2 = u1 * belief2               # (B, K)

    fused_belief = (b1_b2 + b1_u2 + u1_b2) / norm

    # Fused uncertainty
    fused_uncertainty = (u1 * u2) / norm

    return fused_belief, fused_uncertainty


print("✅ Dempster-Shafer Fusion defined.")

## 🏗️ Full Model Assembly: MultimodalEDLCoAttention
Menggabungkan semua modul menjadi satu arsitektur end-to-end:
1. Encode → 2. Decouple → 3. Cross-Attend → 4. EDL Gate → 5. DS Fusion

In [ ]:
# ============================================================
# FULL MODEL (Assembly of all modules)
# ============================================================
class MultimodalEDLCoAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Phase 2: Unimodal Encoders
        self.text_encoder = TextEncoder(
            cfg.TEXT_MODEL, cfg.FEATURE_DIM, cfg.HIDDEN_DIM, cfg.DROPOUT
        )
        self.image_encoder = ImageEncoder(
            cfg.IMAGE_MODEL, cfg.FEATURE_DIM, cfg.HIDDEN_DIM, cfg.DROPOUT
        )

        # Phase 3: Feature Decoupling
        self.text_decoupler = FeatureDecoupler(cfg.HIDDEN_DIM, cfg.DROPOUT)
        self.image_decoupler = FeatureDecoupler(cfg.HIDDEN_DIM, cfg.DROPOUT)

        # Phase 4: Bi-directional Cross-Attention
        self.cross_attention = BiDirectionalCrossAttention(
            cfg.HIDDEN_DIM, cfg.NUM_HEADS, cfg.DROPOUT
        )

        # Phase 5: EDL Layers (one per attention direction)
        self.edl_t2v = EvidentialLayer(cfg.HIDDEN_DIM, cfg.NUM_CLASSES)
        self.edl_v2t = EvidentialLayer(cfg.HIDDEN_DIM, cfg.NUM_CLASSES)

    def freeze_backbones(self):
        self.text_encoder.freeze_backbone()
        self.image_encoder.freeze_backbone()

    def unfreeze_backbones(self):
        self.text_encoder.unfreeze_backbone()
        self.image_encoder.unfreeze_backbone()

    def forward(self, input_ids, attention_mask, pixel_values):
        # ---- Phase 2: Feature Extraction ----
        text_seq, text_mask = self.text_encoder(input_ids, attention_mask)
        image_seq = self.image_encoder(pixel_values)

        # ---- Phase 3: Feature Decoupling ----
        text_sent, text_bg = self.text_decoupler(text_seq)
        image_sent, image_bg = self.image_decoupler(image_seq)

        # ---- Phase 4: Bi-directional Cross-Attention ----
        pooled_t2v, pooled_v2t = self.cross_attention(
            text_sent, image_sent, text_mask
        )

        # ---- Phase 5: EDL & Adaptive Gating ----
        edl_t2v = self.edl_t2v(pooled_t2v)
        edl_v2t = self.edl_v2t(pooled_v2t)

        # Adaptive Gating: suppress uncertain features
        gated_t2v = pooled_t2v * (1.0 - edl_t2v["uncertainty"])
        gated_v2t = pooled_v2t * (1.0 - edl_v2t["uncertainty"])

        # ---- Phase 6: Dempster-Shafer Fusion ----
        fused_belief, fused_uncertainty = dempster_shafer_fusion(
            edl_t2v["belief"], edl_t2v["uncertainty"],
            edl_v2t["belief"], edl_v2t["uncertainty"]
        )

        # Orthogonal loss components
        orth_loss_text = FeatureDecoupler.orthogonal_loss(text_sent, text_bg)
        orth_loss_image = FeatureDecoupler.orthogonal_loss(image_sent, image_bg)

        return {
            "fused_belief": fused_belief,
            "fused_uncertainty": fused_uncertainty,
            "edl_t2v": edl_t2v,
            "edl_v2t": edl_v2t,
            "gated_t2v": gated_t2v,
            "gated_v2t": gated_v2t,
            "orth_loss": orth_loss_text + orth_loss_image,
        }


# ============================================================
# INSTANTIATE & VERIFY MODEL
# ============================================================
model = MultimodalEDLCoAttention(CFG).to(device)
model.freeze_backbones()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")

# Forward pass sanity check
with torch.no_grad():
    test_batch = next(iter(train_loader))
    test_out = model(
        test_batch["input_ids"].to(device),
        test_batch["attention_mask"].to(device),
        test_batch["pixel_values"].to(device)
    )
    print(f"\n✅ Forward pass successful!")
    print(f"  Fused belief shape:       {test_out['fused_belief'].shape}")
    print(f"  Fused uncertainty shape:  {test_out['fused_uncertainty'].shape}")
    print(f"  EDL T2V alpha shape:      {test_out['edl_t2v']['alpha'].shape}")
    print(f"  Orthogonal loss:          {test_out['orth_loss'].item():.4f}")

## 🎯 Fase 7: Joint Loss Function & Optimizer
- **EDL MSE Loss** berbasis distribusi Dirichlet
- **KL Divergence Regularization** dengan annealing
- **Orthogonal Loss** untuk feature decoupling
- **Optimizer:** AdamW dengan differential learning rate

In [ ]:
# ============================================================
# EDL LOSS FUNCTION (Step 7.1 - 7.2)
# ============================================================
def edl_mse_loss(alpha, y_onehot, num_classes):
    """
    EDL Expected Mean Square Error Loss.
    L = sum( (y_k - alpha_k/S)^2 + alpha_k*(S-alpha_k) / (S^2*(S+1)) )
    """
    S = alpha.sum(dim=-1, keepdim=True)         # (B, 1)
    p = alpha / S                                # (B, K) predicted prob

    # MSE term
    mse = (y_onehot - p) ** 2

    # Variance term (epistemic uncertainty in prediction)
    var = alpha * (S - alpha) / (S ** 2 * (S + 1.0))

    loss = (mse + var).sum(dim=-1)               # (B,)
    return loss.mean()


def kl_divergence_reg(alpha, y_onehot, num_classes):
    """
    KL Divergence antara Dir(alpha_tilde) dan Dir(1,1,...,1).
    alpha_tilde = y + (1-y)*(alpha) -> remove evidence for correct class.
    """
    # Remove non-misleading evidence
    alpha_tilde = y_onehot + (1.0 - y_onehot) * alpha

    beta = torch.ones_like(alpha_tilde)  # Uniform Dirichlet
    S_alpha = alpha_tilde.sum(dim=-1, keepdim=True)
    S_beta = beta.sum(dim=-1, keepdim=True)

    ln_alpha = torch.lgamma(S_alpha) - torch.lgamma(alpha_tilde).sum(dim=-1, keepdim=True)
    ln_beta = torch.lgamma(beta).sum(dim=-1, keepdim=True) - torch.lgamma(S_beta)

    dg_term = torch.digamma(alpha_tilde) - torch.digamma(S_alpha)
    kl = ln_alpha + ln_beta + ((alpha_tilde - beta) * dg_term).sum(dim=-1, keepdim=True)

    return kl.mean()


def compute_edl_loss(alpha, labels, num_classes, epoch, annealing_step):
    """Combined EDL loss with KL annealing."""
    y_onehot = F.one_hot(labels, num_classes).float()

    mse_loss = edl_mse_loss(alpha, y_onehot, num_classes)

    # Annealing coefficient (Step 7.2)
    lambda_t = min(1.0, epoch / annealing_step)
    kl_loss = kl_divergence_reg(alpha, y_onehot, num_classes)

    return mse_loss + lambda_t * kl_loss


# ============================================================
# OPTIMIZER & SCHEDULER (Step 7.3)
# ============================================================
def get_optimizer_and_scheduler(model, cfg, num_training_steps):
    """AdamW with differential LR: backbone vs head."""
    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg.LR_BACKBONE},
        {"params": head_params, "lr": cfg.LR_HEAD},
    ], weight_decay=cfg.WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_training_steps, eta_min=1e-7
    )

    return optimizer, scheduler


num_training_steps = len(train_loader) * CFG.EPOCHS
optimizer, scheduler = get_optimizer_and_scheduler(model, CFG, num_training_steps)

print(f"✅ Optimizer & Scheduler configured.")
print(f"  Total training steps: {num_training_steps}")
print(f"  LR backbone: {CFG.LR_BACKBONE}")
print(f"  LR head: {CFG.LR_HEAD}")
print(f"  Annealing step: {CFG.ANNEALING_STEP}")

## 📈 Fase 8: Training Loop & Evaluasi
- Training loop standar PyTorch
- Evaluasi: Accuracy, Macro-F1, rata-rata uncertainty pada data salah klasifikasi
- Best model saving berdasarkan Val F1

In [ ]:
# ============================================================
# TRAINING & EVALUATION FUNCTIONS (Step 8.1 - 8.2)
# ============================================================
from tqdm.auto import tqdm

def train_one_epoch(model, dataloader, optimizer, scheduler, epoch, cfg):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    pbar = tqdm(dataloader, desc=f"Train Epoch {epoch+1}/{cfg.EPOCHS}")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask, pixel_values)

        # EDL Loss for both directions
        loss_t2v = compute_edl_loss(
            outputs["edl_t2v"]["alpha"], labels,
            cfg.NUM_CLASSES, epoch, cfg.ANNEALING_STEP
        )
        loss_v2t = compute_edl_loss(
            outputs["edl_v2t"]["alpha"], labels,
            cfg.NUM_CLASSES, epoch, cfg.ANNEALING_STEP
        )

        # Orthogonal regularization
        orth_loss = outputs["orth_loss"]

        # Total loss
        loss = loss_t2v + loss_v2t + cfg.LAMBDA_ORTH * orth_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Predictions from fused belief
        preds = outputs["fused_belief"].argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, dataloader, epoch, cfg):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    all_uncertainties = []
    correct_mask = []

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask, pixel_values)

        loss_t2v = compute_edl_loss(
            outputs["edl_t2v"]["alpha"], labels,
            cfg.NUM_CLASSES, epoch, cfg.ANNEALING_STEP
        )
        loss_v2t = compute_edl_loss(
            outputs["edl_v2t"]["alpha"], labels,
            cfg.NUM_CLASSES, epoch, cfg.ANNEALING_STEP
        )
        orth_loss = outputs["orth_loss"]
        loss = loss_t2v + loss_v2t + cfg.LAMBDA_ORTH * orth_loss
        total_loss += loss.item()

        preds = outputs["fused_belief"].argmax(dim=-1).cpu().numpy()
        uncertainties = outputs["fused_uncertainty"].squeeze(-1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        all_uncertainties.extend(uncertainties)
        correct_mask.extend(preds == labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")

    # Average uncertainty on misclassified samples
    all_uncertainties = np.array(all_uncertainties)
    correct_mask = np.array(correct_mask)
    avg_u_correct = all_uncertainties[correct_mask].mean() if correct_mask.sum() > 0 else 0
    avg_u_wrong = all_uncertainties[~correct_mask].mean() if (~correct_mask).sum() > 0 else 0

    return avg_loss, acc, f1, avg_u_correct, avg_u_wrong


print("✅ Training & Evaluation functions defined.")

In [ ]:
# ============================================================
# EXECUTE TRAINING (Step 8.1 - 8.3)
# ============================================================
best_val_f1 = 0
best_model_state = None
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [],
           "train_f1": [], "val_f1": [], "val_u_correct": [], "val_u_wrong": []}

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

for epoch in range(CFG.EPOCHS):
    # Unfreeze backbone after FREEZE_EPOCHS
    if epoch == CFG.FREEZE_EPOCHS:
        model.unfreeze_backbones()
        # Re-create optimizer to include backbone params
        optimizer, scheduler = get_optimizer_and_scheduler(
            model, CFG, len(train_loader) * (CFG.EPOCHS - epoch)
        )
        print(f"\n🔓 Backbone UNFROZEN at epoch {epoch+1}!")
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"   Trainable params: {trainable:,}\n")

    # Train
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, scheduler, epoch, CFG
    )

    # Validate
    val_loss, val_acc, val_f1, val_u_correct, val_u_wrong = evaluate(
        model, val_loader, epoch, CFG
    )

    # Log history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    history["val_u_correct"].append(val_u_correct)
    history["val_u_wrong"].append(val_u_wrong)

    print(f"\nEpoch {epoch+1}/{CFG.EPOCHS}")
    print(f"  Train - Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print(f"  Val Uncertainty - Correct: {val_u_correct:.4f} | Wrong: {val_u_wrong:.4f}")

    # Save best model (Step 8.3)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"  ⭐ New best model! Val F1: {val_f1:.4f}")

print("\n" + "=" * 70)
print(f"TRAINING COMPLETE! Best Val F1: {best_val_f1:.4f}")
print("=" * 70)

In [ ]:
# ============================================================
# FINAL EVALUATION ON TEST SET
# ============================================================
# Load best model
model.load_state_dict(best_model_state)
model.eval()

test_loss, test_acc, test_f1, test_u_correct, test_u_wrong = evaluate(
    model, test_loader, CFG.EPOCHS - 1, CFG
)

print("=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)
print(f"  Test Accuracy:    {test_acc:.4f}")
print(f"  Test Macro-F1:    {test_f1:.4f}")
print(f"  Test Loss:        {test_loss:.4f}")
print(f"  Avg Uncertainty (Correct):   {test_u_correct:.4f}")
print(f"  Avg Uncertainty (Wrong):     {test_u_wrong:.4f}")
print("=" * 70)

# Detailed classification report
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        outputs = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["pixel_values"].to(device)
        )
        preds = outputs["fused_belief"].argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["label"].numpy())

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["Negative", "Neutral", "Positive"]))

# Save model
torch.save(best_model_state, "best_edl_coattn_model.pt")
print("✅ Model saved to best_edl_coattn_model.pt")